# Metrics Module Test Notebook

Validates `cuic_quant.metrics` against backtester-compatible output.


In [2]:
import sys
from pathlib import Path

# Add src to path so cuic_quant can be imported
project_root = Path.cwd()
if project_root.name == 'tools':
    project_root = project_root.parent
src_path = project_root / 'src'
if str(src_path) not in sys.path:
    sys.path.insert(0, str(src_path))

import pandas as pd

from cuic_quant.metrics import (
    calculate_all_metrics,
    calculate_max_drawdown,
    calculate_profit_factor,
    calculate_sharpe_ratio,
    calculate_win_rate,
)

In [3]:
REQUIRED_BACKTESTER_COLUMNS = [
    'timestamp', 'game', 'action', 'bet_size',
    'odds', 'outcome', 'pnl', 'cumulative_pnl', 'bankroll',
]

# Use project_root from setup cell
candidate_paths = [
    project_root / 'data' / 'backtest_results.csv',
    project_root / 'data' / 'dummy_backtest_output.csv',
]

selected_path = next((p for p in candidate_paths if p.exists()), None)
if selected_path is None:
    raise FileNotFoundError(
        'No backtester output found. Expected one of: ' + ', '.join(str(p) for p in candidate_paths)
    )

results = pd.read_csv(selected_path)
print(f'Loaded {len(results)} rows from {selected_path}')
print('Columns:', results.columns.tolist())

Loaded 25 rows from C:\Users\james\PycharmProjects\PythonProject\CUIC_Sem2_Project\data\dummy_backtest_output.csv
Columns: ['timestamp', 'game', 'action', 'bet_size', 'odds', 'outcome', 'pnl', 'cumulative_pnl', 'bankroll']


In [4]:
missing = [c for c in REQUIRED_BACKTESTER_COLUMNS if c not in results.columns]
assert not missing, f'Missing expected backtester columns: {missing}'

metrics = calculate_all_metrics(results)
metrics


{'total_trades': 25,
 'win_rate': 0.6,
 'total_pnl': 305.0,
 'sharpe_ratio': 2.0568874347099158,
 'max_drawdown': 0.75,
 'profit_factor': 1.305}

In [5]:
# Individual metric checks
print('Sharpe:', calculate_sharpe_ratio(results['pnl']))
print('Max drawdown:', calculate_max_drawdown(results['cumulative_pnl']))
print('Win rate:', calculate_win_rate(results['outcome']))
print('Profit factor:', calculate_profit_factor(results['pnl']))


Sharpe: 2.0568874347099158
Max drawdown: 0.75
Win rate: 0.6
Profit factor: 1.305


In [6]:
# Edge-case checks from brief
assert calculate_sharpe_ratio(pd.Series(dtype=float)) == 0.0
assert calculate_profit_factor(pd.Series([1.0, 2.0])) == float('inf')
assert calculate_profit_factor(pd.Series([-1.0, -2.0])) == 0.0
print('Edge-case checks passed')


Edge-case checks passed
